# Inspect 10.5 T water and lipid simulation resources

This notebook automatically discovers all 10.5 T `SimulationResources_water_lipid_v1.h5` files and plots 20 water and 20 lipid spectra per subject. Selection is reproducible through a fixed random seed.

In [ ]:
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np

DATA_ROOT = Path('../data/10.5T').resolve()
RESOURCE_PATTERN = '*/TrainData/SimulationResources_water_lipid_v1.h5'
N_EXAMPLES = 20
RANDOM_SEED = 42
PPM_REFERENCE = 4.65
PPM_LIMITS = (7.0, 0.0)
NORMALIZE_EACH_SPECTRUM = True

resource_paths = sorted(DATA_ROOT.glob(RESOURCE_PATTERN))
if not resource_paths:
    raise FileNotFoundError(
        f'No resources matching {RESOURCE_PATTERN!r} below {DATA_ROOT}'
    )

print(f'Found {len(resource_paths)} subject(s):')
for path in resource_paths:
    print(f'  {path.parent.parent.name}: {path}')


In [ ]:
def valid_fids(fids):
    fids = np.asarray(fids)
    valid = np.isfinite(fids).all(axis=-1) & (np.max(np.abs(fids), axis=-1) > 0)
    return fids[valid], np.flatnonzero(valid)


def select_examples(fids, n_examples, rng):
    if len(fids) < n_examples:
        raise ValueError(
            f'Only {len(fids)} valid FIDs are available; requested {n_examples}.'
        )
    selected = np.sort(rng.choice(len(fids), size=n_examples, replace=False))
    return fids[selected], selected


def spectra_and_ppm(fids, bandwidth_hz, nmr_frequency_hz):
    spectra = np.fft.fftshift(np.fft.fft(fids, axis=-1), axes=-1)
    frequencies_hz = np.fft.fftshift(
        np.fft.fftfreq(fids.shape[-1], d=1.0 / bandwidth_hz)
    )
    ppm = PPM_REFERENCE - frequencies_hz / (nmr_frequency_hz / 1e6)

    scales = np.max(np.abs(spectra), axis=-1, keepdims=True)
    if NORMALIZE_EACH_SPECTRUM:
        spectra = spectra / np.maximum(scales, np.finfo(float).eps)
    return spectra, ppm, scales[:, 0]


def plot_twenty(spectra, ppm, scales, title, identifiers):
    figure, axes = plt.subplots(
        4, 5, figsize=(15, 9), sharex=True, sharey=NORMALIZE_EACH_SPECTRUM
    )
    for index, (axis, spectrum, identifier, scale) in enumerate(
        zip(axes.flat, spectra, identifiers, scales)
    ):
        axis.plot(ppm, spectrum.real, color='black', linewidth=0.9, label='Real')
        axis.plot(
            ppm, np.abs(spectrum), color='tab:orange', linewidth=0.7,
            alpha=0.75, label='Magnitude'
        )
        axis.set_xlim(*PPM_LIMITS)
        if NORMALIZE_EACH_SPECTRUM:
            axis.set_ylim(-1.05, 1.05)
        axis.grid(alpha=0.18, linewidth=0.5)
        axis.set_title(f'{index + 1}: {identifier}\nscale={scale:.3g}', fontsize=8)
        if index == 0:
            axis.legend(loc='upper left', fontsize=7)

    for axis in axes[-1, :]:
        axis.set_xlabel('ppm')
    for axis in axes[:, 0]:
        axis.set_ylabel('Normalized amplitude' if NORMALIZE_EACH_SPECTRUM else 'Amplitude [a.u.]')
    figure.suptitle(title, fontsize=14)
    figure.tight_layout(rect=(0, 0, 1, 0.97))
    plt.show()
    return figure


In [ ]:
figures = {}

for subject_index, resource_path in enumerate(resource_paths):
    subject = resource_path.parent.parent.name
    rng = np.random.default_rng(RANDOM_SEED + subject_index)

    with h5py.File(resource_path, 'r') as resources:
        water_volume = np.asarray(resources['water_fids'], dtype=np.complex64)
        brain_mask = np.asarray(resources['brain_mask'], dtype=bool)
        lipid_pool = np.asarray(resources['lipid_fids'], dtype=np.complex64)
        bandwidth_hz = float(resources.attrs['bandwidth_hz'])
        nmr_frequency_hz = float(resources.attrs.get('nmr_frequency_hz', 447053391.0))

    if water_volume.shape[:-1] != brain_mask.shape:
        raise ValueError(
            f'{subject}: water shape {water_volume.shape} and brain mask '
            f'{brain_mask.shape} are inconsistent.'
        )

    brain_coordinates = np.argwhere(brain_mask)
    water_pool, valid_water_indices = valid_fids(water_volume[brain_mask])
    lipid_pool, valid_lipid_indices = valid_fids(lipid_pool)

    selected_water, water_selection = select_examples(water_pool, N_EXAMPLES, rng)
    selected_lipids, lipid_selection = select_examples(lipid_pool, N_EXAMPLES, rng)

    water_coordinates = brain_coordinates[valid_water_indices[water_selection]]
    lipid_ids = valid_lipid_indices[lipid_selection]
    water_identifiers = [
        f'voxel ({x}, {y}, {z})' for x, y, z in water_coordinates
    ]
    lipid_identifiers = [f'pool index {index}' for index in lipid_ids]

    water_spectra, ppm, water_scales = spectra_and_ppm(
        selected_water, bandwidth_hz, nmr_frequency_hz
    )
    lipid_spectra, lipid_ppm, lipid_scales = spectra_and_ppm(
        selected_lipids, bandwidth_hz, nmr_frequency_hz
    )

    print(
        f'\n{subject}: water pool={len(water_pool):,}, '
        f'lipid pool={len(lipid_pool):,}, points={water_volume.shape[-1]}, '
        f'bandwidth={bandwidth_hz:g} Hz'
    )
    figures[(subject, 'water')] = plot_twenty(
        water_spectra, ppm, water_scales,
        f'{subject}: 20 isolated water spectra', water_identifiers
    )
    figures[(subject, 'lipid')] = plot_twenty(
        lipid_spectra, lipid_ppm, lipid_scales,
        f'{subject}: 20 water-removed lipid spectra', lipid_identifiers
    )

print(f'\nCreated {len(figures)} figures.')
